# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/32-PythonTransferLearning.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 32 - Python ile Transfer Learning ve Önceden Eğitilmiş Modeller

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bir önceki derste CNN ile görüntülerden özellik öğrenen bir sınıflandırma modeli geliştirdik.

Bu derste önemli bir soruya cevap arayacağız:

**Her yeni görüntü problemi için bütün CNN'i sıfırdan eğitmek zorunda mıyız?**

Cevap çoğu zaman hayırdır.

Daha önce başka bir problem üzerinde eğitilmiş bir modelin öğrendiği özelliklerden yararlanabiliriz. Bu yaklaşıma **Transfer Learning** denir.

Bu derste iki farklı uygulama yapacağız:

1. İnternet bağlantısı gerektirmeyen yerel bir transfer learning deneyi.
2. ImageNet ağırlıklarıyla MobileNetV2 kullanarak gerçek önceden eğitilmiş model yaklaşımı.

Bu dersin sonunda öğrencinin:

- transfer learning kavramını açıklayabilmesi,
- source task ve target task kavramlarını ayırabilmesi,
- feature extractor oluşturabilmesi,
- katmanları dondurabilmesi,
- `trainable=False` kullanabilmesi,
- yeni classification head ekleyebilmesi,
- feature extraction yapabilmesi,
- sıfırdan eğitim ile transfer learning'i karşılaştırabilmesi,
- fine-tuning uygulayabilmesi,
- düşük learning rate kullanımının nedenini açıklayabilmesi,
- MobileNetV2 gibi önceden eğitilmiş modelleri kullanabilmesi,
- modele özel ön işleme uygulayabilmesi,
- BatchNormalization katmanlarına fine-tuning sırasında dikkat edebilmesi,
- transfer modelini kaydedip yeniden yükleyebilmesi

hedeflenmektedir.


# 1. Transfer Learning Nedir?

Transfer Learning, bir modelin daha önce öğrendiği özellikleri başka fakat ilişkili bir problemde yeniden kullanmaktır.

Basit örnek:

```text
Büyük görüntü veri kümesi
↓
Kenarları, dokuları, şekilleri öğrenmiş CNN
↓
Öğrenilmiş özellikleri koru
↓
Kendi küçük veri kümem
↓
Yeni sınıflandırıcı
```

Model bütün görsel özellikleri sıfırdan öğrenmek yerine daha önce öğrenilmiş temsillerden yararlanır.


# 2. Source Task ve Target Task

Transfer learning iki problem arasında gerçekleşir.

### Source Task

Modelin ilk eğitildiği problem.

Örnek:

```text
0-9 rakam sınıflandırma
```

### Target Task

Öğrenilmiş özelliklerin aktarıldığı yeni problem.

Örnek:

```text
Tek / Çift rakam sınıflandırma
```


# 3. Neden Transfer Learning Kullanılır?

Transfer learning özellikle:

- veri az olduğunda,
- büyük modeli sıfırdan eğitmek pahalı olduğunda,
- source ve target problemleri ilişkili olduğunda,
- önceden eğitilmiş güçlü modeller bulunduğunda

yararlı olabilir.

Görüntü alanında ImageNet üzerinde eğitilmiş modeller bu amaçla sık kullanılır.


# 4. Temel Transfer Learning Akışı

```text
Önceden Eğitilmiş Model
↓
Feature Extractor Bölümünü Al
↓
Katmanları Dondur
↓
Yeni Classification Head Ekle
↓
Sadece Yeni Katmanları Eğit
↓
Gerekirse Fine-Tuning
```


# 5. Feature Extraction ve Fine-Tuning Farkı

### Feature Extraction

Önceden eğitilmiş taban model dondurulur.

```text
Base Model → trainable=False
Yeni Head  → trainable=True
```

Sadece yeni katmanlar öğrenir.

### Fine-Tuning

İlk eğitim tamamlandıktan sonra taban modelin bazı katmanları açılır ve çok küçük learning rate ile yeniden eğitilir.


# 6. Önce Feature Extraction

Fine-tuning'e doğrudan başlamamak önemlidir.

Yeni eklediğimiz classification head başlangıçta rastgele ağırlıklara sahiptir.

Önce:

```text
Base Model Dondur
↓
Yeni Head Eğit
```

ardından gerekiyorsa:

```text
Bazı Base Katmanlarını Aç
↓
Düşük Learning Rate
↓
Fine-Tune
```

yaklaşımı kullanılır.


# 7. Gerekli Kütüphaneler

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(
    "TensorFlow:",
    tf.__version__
)

print(
    "Keras:",
    keras.__version__
)


# 8. Tekrar Üretilebilirlik

In [ ]:
keras.utils.set_random_seed(
    42
)


# 9. İlk Uygulama: Yerel Transfer Learning

İlk transfer learning deneyimiz tamamen yerel çalışacaktır.

İnternet bağlantısına ihtiyaç duymaz.

Scikit-learn Digits veri kümesini kullanacağız.

Source task:

```text
0-9 rakam sınıflandırma
```

Target task:

```text
Tek / Çift rakam sınıflandırma
```


# 10. Digits Veri Kümesini Yüklemek

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()

images = digits.images.astype(
    np.float32
)

labels = digits.target

print(
    images.shape,
    labels.shape
)


# 11. Piksel Normalizasyonu

In [ ]:
images = (
    images
    /
    16.0
)

print(
    images.min(),
    images.max()
)


# 12. CNN Kanal Boyutu

In [ ]:
X = np.expand_dims(
    images,
    axis=-1
)

y = labels

print(
    X.shape
)


# 13. Source ve Transfer Havuzu Ayırmak

Önce veri kümesinin bir bölümünü source model eğitiminde kullanacağız.

Kalan örnekleri daha sonra target problem için kullanacağız.


In [ ]:
from sklearn.model_selection import train_test_split

X_source, X_transfer_pool, y_source, y_transfer_pool = train_test_split(
    X,
    y,
    test_size=0.35,
    random_state=42,
    stratify=y
)

print(
    "Source eğitim havuzu:",
    X_source.shape
)

print(
    "Transfer havuzu:",
    X_transfer_pool.shape
)


Transfer havuzundaki örnekler source CNN'in eğitiminde kullanılmayacaktır.

Böylece target problemde yeni örnekler üzerinde çalışabiliriz.


# 14. Feature Extractor Oluşturmak

CNN'in görüntüden özellik çıkaran bölümünü ayrı bir model olarak tanımlayalım.


In [ ]:
keras.utils.set_random_seed(
    42
)

feature_extractor = keras.Sequential(
    [
        keras.Input(
            shape=(
                8,
                8,
                1
            )
        ),
        layers.Conv2D(
            16,
            3,
            padding="same",
            activation="relu"
        ),
        layers.MaxPooling2D(
            2
        ),
        layers.Conv2D(
            32,
            3,
            padding="same",
            activation="relu"
        ),
        layers.MaxPooling2D(
            2
        )
    ],
    name="feature_extractor"
)

feature_extractor.summary()


Bu model henüz sınıf tahmini yapmaz.

Görevi:

```text
Görüntü
↓
Convolution
↓
Feature Maps
```

üretmektir.


# 15. Source Classification Head

Feature extractor'ın üzerine 10 rakam sınıfı için bir classification head ekleyelim.


In [ ]:
source_model = keras.Sequential([
    feature_extractor,
    layers.Flatten(),
    layers.Dense(
        64,
        activation="relu"
    ),
    layers.Dropout(
        0.20
    ),
    layers.Dense(
        10,
        activation="softmax"
    )
])

source_model.summary()


# 16. Source Modeli Compile Etmek

In [ ]:
source_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)


# 17. Source Modeli Eğitmek

In [ ]:
source_early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

source_history = source_model.fit(
    X_source,
    y_source,
    validation_split=0.20,
    epochs=30,
    batch_size=32,
    callbacks=[
        source_early_stop
    ],
    verbose=0
)

print(
    "Epoch:",
    len(
        source_history.history[
            "loss"
        ]
    )
)


# 18. Source Learning Curve

In [ ]:
plt.plot(
    source_history.history[
        "accuracy"
    ],
    label="Train"
)

plt.plot(
    source_history.history[
        "val_accuracy"
    ],
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Source Rakam Modeli")
plt.legend()
plt.show()


# 19. Source Modelin Transfer Havuzundaki Başarısı

In [ ]:
source_loss, source_accuracy = (
    source_model.evaluate(
        X_transfer_pool,
        y_transfer_pool,
        verbose=0
    )
)

print(
    "Source accuracy:",
    source_accuracy
)


Bu model 0-9 rakamlarını tanımayı öğrenmiştir.

Şimdi classification head'i değiştireceğiz fakat feature extractor'ın öğrendiği görsel özellikleri kullanmaya devam edeceğiz.


# 20. Öğrenilmiş Feature Extractor

Source model eğitildiği için `feature_extractor` içindeki convolution ağırlıkları artık rastgele değildir.

Source problemden öğrenilmiş ağırlıklardır.


In [ ]:
print(
    "Feature extractor parametre:",
    feature_extractor.count_params()
)


# 21. Feature Map Üretmek

In [ ]:
ornek_feature = (
    feature_extractor.predict(
        X_transfer_pool[:1],
        verbose=0
    )
)

print(
    ornek_feature.shape
)


# 22. Source Görüntü

In [ ]:
plt.imshow(
    X_transfer_pool[
        0,
        :,
        :,
        0
    ],
    cmap="gray"
)

plt.title(
    f"Rakam: {y_transfer_pool[0]}"
)

plt.axis("off")
plt.show()


# 23. Öğrenilmiş Feature Map

In [ ]:
plt.imshow(
    ornek_feature[
        0,
        :,
        :,
        0
    ],
    cmap="gray"
)

plt.title("Öğrenilmiş Feature Map")
plt.axis("off")
plt.show()


# 24. Target Task: Tek / Çift

Şimdi hedefi değiştirelim.

```text
0 → çift
1 → tek
2 → çift
3 → tek
...
```

Target etiket:

```text
0 → çift
1 → tek
```


In [ ]:
y_binary = (
    y_transfer_pool
    %
    2
).astype(
    np.int32
)

print(
    y_transfer_pool[:15]
)

print(
    y_binary[:15]
)


# 25. Target Train-Test Ayrımı

In [ ]:
X_target_train, X_target_test, y_target_train, y_target_test = train_test_split(
    X_transfer_pool,
    y_binary,
    test_size=0.45,
    random_state=42,
    stratify=y_binary
)

print(
    X_target_train.shape,
    X_target_test.shape
)


# 26. Küçük Veri Senaryosu

Transfer learning'in veri az olduğunda avantajını görmek için target eğitim verisinin yalnızca bir bölümünü kullanalım.


In [ ]:
rng = np.random.default_rng(
    42
)

secili_indexler = []

for sinif in [
    0,
    1
]:
    sinif_index = np.where(
        y_target_train
        ==
        sinif
    )[0]

    secim = rng.choice(
        sinif_index,
        size=min(
            100,
            len(
                sinif_index
            )
        ),
        replace=False
    )

    secili_indexler.extend(
        secim.tolist()
    )

secili_indexler = np.array(
    secili_indexler
)

X_target_small = (
    X_target_train[
        secili_indexler
    ]
)

y_target_small = (
    y_target_train[
        secili_indexler
    ]
)

print(
    X_target_small.shape
)


Yaklaşık 200 target eğitim örneğiyle çalışıyoruz.


# 27. Feature Extractor'ı Dondurmak

Transfer learning'in temel adımı:


In [ ]:
feature_extractor.trainable = False

print(
    "Feature extractor trainable:",
    feature_extractor.trainable
)


# 28. Trainable ve Non-Trainable Weight Sayısı

In [ ]:
print(
    "Trainable weights:",
    len(
        feature_extractor.trainable_weights
    )
)

print(
    "Non-trainable weights:",
    len(
        feature_extractor.non_trainable_weights
    )
)


Dondurulduğunda convolution ağırlıkları target eğitimi sırasında değiştirilmez.


# 29. Yeni Target Head

Eski 10 sınıflı head'i kullanmayacağız.

Yeni problem:

```text
tek / çift
```

olduğu için yeni sigmoid head ekleyeceğiz.


In [ ]:
keras.utils.set_random_seed(
    42
)

target_model = keras.Sequential([
    keras.Input(
        shape=(
            8,
            8,
            1
        )
    ),
    feature_extractor,
    layers.Flatten(),
    layers.Dense(
        32,
        activation="relu"
    ),
    layers.Dropout(
        0.20
    ),
    layers.Dense(
        1,
        activation="sigmoid"
    )
])

target_model.summary()


# 30. Trainable Parametreleri Görmek

In [ ]:
print(
    "Toplam parametre:",
    target_model.count_params()
)

print(
    "Trainable parametre:",
    sum(
        int(
            np.prod(
                w.shape
            )
        )
        for w
        in target_model.trainable_weights
    )
)

print(
    "Non-trainable parametre:",
    sum(
        int(
            np.prod(
                w.shape
            )
        )
        for w
        in target_model.non_trainable_weights
    )
)


Target eğitimin ilk aşamasında yalnızca yeni eklenen Dense katmanlar öğrenmektedir.


# 31. Target Modeli Compile Etmek

In [ ]:
target_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)


# 32. Frozen Base ile Eğitim

In [ ]:
target_early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

target_history = target_model.fit(
    X_target_small,
    y_target_small,
    validation_split=0.20,
    epochs=40,
    batch_size=16,
    callbacks=[
        target_early_stop
    ],
    verbose=0
)

print(
    "Epoch:",
    len(
        target_history.history[
            "loss"
        ]
    )
)


# 33. Target Learning Curve

In [ ]:
plt.plot(
    target_history.history[
        "accuracy"
    ],
    label="Train"
)

plt.plot(
    target_history.history[
        "val_accuracy"
    ],
    label="Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Transfer Learning - Frozen Base")
plt.legend()
plt.show()


# 34. Target Test Değerlendirmesi

In [ ]:
transfer_loss, transfer_accuracy = (
    target_model.evaluate(
        X_target_test,
        y_target_test,
        verbose=0
    )
)

print(
    "Transfer Learning Accuracy:",
    transfer_accuracy
)


# 35. Test Tahminleri

In [ ]:
transfer_prob = (
    target_model.predict(
        X_target_test,
        verbose=0
    )
    .reshape(-1)
)

transfer_pred = (
    transfer_prob
    >=
    0.5
).astype(
    int
)


# 36. Classification Report

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print(
    classification_report(
        y_target_test,
        transfer_pred,
        target_names=[
            "Çift",
            "Tek"
        ]
    )
)


# 37. Confusion Matrix

In [ ]:
transfer_cm = confusion_matrix(
    y_target_test,
    transfer_pred
)

ConfusionMatrixDisplay(
    confusion_matrix=transfer_cm,
    display_labels=[
        "Çift",
        "Tek"
    ]
).plot()

plt.title(
    "Transfer Learning Confusion Matrix"
)

plt.show()


# 38. Sıfırdan Model Karşılaştırması

Aynı küçük target veri kümesini kullanarak tamamen rastgele ağırlıklardan başlayan aynı yapıda bir CNN oluşturalım.


In [ ]:
keras.utils.set_random_seed(
    42
)

scratch_model = keras.Sequential([
    keras.Input(
        shape=(
            8,
            8,
            1
        )
    ),
    layers.Conv2D(
        16,
        3,
        padding="same",
        activation="relu"
    ),
    layers.MaxPooling2D(
        2
    ),
    layers.Conv2D(
        32,
        3,
        padding="same",
        activation="relu"
    ),
    layers.MaxPooling2D(
        2
    ),
    layers.Flatten(),
    layers.Dense(
        32,
        activation="relu"
    ),
    layers.Dropout(
        0.20
    ),
    layers.Dense(
        1,
        activation="sigmoid"
    )
])

scratch_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)


# 39. Sıfırdan Modeli Eğitmek

In [ ]:
scratch_history = scratch_model.fit(
    X_target_small,
    y_target_small,
    validation_split=0.20,
    epochs=40,
    batch_size=16,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ],
    verbose=0
)

scratch_loss, scratch_accuracy = (
    scratch_model.evaluate(
        X_target_test,
        y_target_test,
        verbose=0
    )
)

print(
    "Scratch Accuracy:",
    scratch_accuracy
)


# 40. Transfer Learning ve Scratch Karşılaştırması

In [ ]:
karsilastirma = pd.DataFrame({
    "Model": [
        "Transfer Learning",
        "Sıfırdan CNN"
    ],
    "TestAccuracy": [
        transfer_accuracy,
        scratch_accuracy
    ]
})

karsilastirma


Transfer learning'in her deneyde mutlaka daha yüksek skor vermesi garanti değildir.

Başarı:

- source ve target problemlerinin ilişkisine,
- source model kalitesine,
- target veri miktarına,
- mimariye,
- eğitim ayarlarına

bağlıdır.


# 41. Feature Extraction'ı Önceden Hesaplamak

Dondurulmuş feature extractor her epoch'ta değişmediği için özellikleri bir kez hesaplayıp ayrı olarak saklamak mümkündür.


In [ ]:
train_features = (
    feature_extractor.predict(
        X_target_small,
        verbose=0
    )
)

test_features = (
    feature_extractor.predict(
        X_target_test,
        verbose=0
    )
)

print(
    train_features.shape
)

print(
    test_features.shape
)


# 42. Feature Vektörlerini Düzleştirmek

In [ ]:
train_features_flat = (
    train_features.reshape(
        len(
            train_features
        ),
        -1
    )
)

test_features_flat = (
    test_features.reshape(
        len(
            test_features
        ),
        -1
    )
)

print(
    train_features_flat.shape
)


# 43. Öğrenilmiş Feature Üzerinde Logistic Regression

CNN feature extractor'ın ürettiği özellikleri klasik makine öğrenmesi modeline bile verebiliriz.


In [ ]:
from sklearn.linear_model import LogisticRegression

feature_lr = LogisticRegression(
    max_iter=1500,
    random_state=42
)

feature_lr.fit(
    train_features_flat,
    y_target_small
)

feature_lr_pred = (
    feature_lr.predict(
        test_features_flat
    )
)

print(
    "Feature Extraction + LR:",
    accuracy_score(
        y_target_test,
        feature_lr_pred
    )
)


Bu yaklaşımda base CNN bütün target verisini yalnızca bir kez işler.

Feature'lar kaydedildikten sonra daha küçük modellerle hızlı deneyler yapılabilir.


# 44. Fine-Tuning'e Geçiş

Şimdi frozen-base aşaması tamamlandı.

Artık feature extractor'ın son katmanlarının bir bölümünü yeniden eğitebiliriz.

Bu aşama **fine-tuning** olarak adlandırılır.


# 45. Fine-Tuning Öncesi Kritik Kural

`trainable` özelliğini değiştirdikten sonra modeli **yeniden compile etmek gerekir**.

Aksi halde eğitim yapılandırması eski trainable durumunu kullanabilir.


# 46. Feature Extractor Katmanlarını Görmek

In [ ]:
for i, layer in enumerate(
    feature_extractor.layers
):
    print(
        i,
        layer.name,
        layer.trainable
    )


# 47. Base Modeli Açmak

Önce tüm feature extractor'ı trainable yapalım.


In [ ]:
feature_extractor.trainable = True


# 48. Yalnızca Son Katmanları Açık Bırakmak

İlk katmanları dondurup son convolution bölümünü fine-tune edeceğiz.


In [ ]:
for layer in feature_extractor.layers[:-2]:
    layer.trainable = False

for i, layer in enumerate(
    feature_extractor.layers
):
    print(
        i,
        layer.name,
        layer.trainable
    )


Bu küçük yerel modelde son iki katmanı açık bırakıyoruz.

Daha büyük modellerde kaç katmanın açılacağı deneysel olarak belirlenir.


# 49. Çok Düşük Learning Rate ile Yeniden Compile

In [ ]:
target_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)


Fine-tuning sırasında önceden öğrenilmiş yararlı ağırlıkları hızlı biçimde bozmamak için learning rate'i belirgin biçimde düşürdük.


# 50. Fine-Tuning Eğitimi

In [ ]:
fine_history = target_model.fit(
    X_target_small,
    y_target_small,
    validation_split=0.20,
    epochs=12,
    batch_size=16,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True
        )
    ],
    verbose=0
)

print(
    "Fine-tuning epoch:",
    len(
        fine_history.history[
            "loss"
        ]
    )
)


# 51. Fine-Tuning Sonrası Test

In [ ]:
fine_loss, fine_accuracy = (
    target_model.evaluate(
        X_target_test,
        y_target_test,
        verbose=0
    )
)

print(
    "Frozen Base Accuracy:",
    transfer_accuracy
)

print(
    "Fine-Tuned Accuracy:",
    fine_accuracy
)


Fine-tuning küçük bir iyileşme sağlayabilir, hiç iyileştirmeyebilir veya overfitting nedeniyle sonucu kötüleştirebilir.

Bu nedenle final kararı validation ve test sonuçlarına göre dikkatle değerlendirmek gerekir.


# 52. Transfer Learning Aşamalarını Karşılaştırmak

In [ ]:
asama_df = pd.DataFrame({
    "Asama": [
        "Sıfırdan CNN",
        "Frozen Transfer",
        "Fine-Tuned Transfer"
    ],
    "TestAccuracy": [
        scratch_accuracy,
        transfer_accuracy,
        fine_accuracy
    ]
})

asama_df


# 53. Karşılaştırma Grafiği

In [ ]:
plt.bar(
    asama_df["Asama"],
    asama_df[
        "TestAccuracy"
    ]
)

plt.ylim(0, 1)
plt.ylabel("Test Accuracy")
plt.title("Transfer Learning Aşamaları")
plt.xticks(rotation=20)
plt.show()


# 54. Öğrenilmiş Base Modeli Kaydetmek

In [ ]:
source_model.save(
    "32-source-rakam-modeli.keras"
)

target_model.save(
    "32-transfer-tek-cift-modeli.keras"
)

print(
    "Modeller kaydedildi."
)


# 55. Target Modeli Yeniden Yüklemek

In [ ]:
transfer_yuklu = (
    keras.models.load_model(
        "32-transfer-tek-cift-modeli.keras"
    )
)

transfer_yuklu.summary()


# 56. Yüklenen Modelle Tahmin

In [ ]:
ornek = X_target_test[:1]

p = transfer_yuklu.predict(
    ornek,
    verbose=0
)[0, 0]

etiket = (
    "Tek"
    if p >= 0.5
    else "Çift"
)

print(
    "Gerçek:",
    "Tek"
    if y_target_test[0] == 1
    else "Çift"
)

print(
    "Tahmin:",
    etiket
)

print(
    "Sigmoid çıktısı:",
    float(p)
)


# 57. İlk Uygulamadan Çıkardığımız Sonuç

Transfer learning'in temel mantığını internet gerektirmeden uyguladık:

```text
Source Task
0-9 sınıflandırma
↓
CNN özellikleri öğrendi
↓
Feature Extractor'ı koruduk
↓
Yeni Target Task
Tek / Çift
↓
Yeni Head
↓
Fine-Tuning
```


# 58. Şimdi Gerçek Önceden Eğitilmiş Modele Geçelim

Gerçek görüntü projelerinde çoğunlukla source modeli kendimiz eğitmek yerine büyük veri kümelerinde önceden eğitilmiş modeller kullanırız.

Keras Applications içinde örnek modeller:

- MobileNetV2,
- MobileNetV3,
- ResNet,
- EfficientNet,
- DenseNet,
- Xception.

Bu derste **MobileNetV2** kullanacağız.


# 59. ImageNet Nedir?

ImageNet, çok sayıda gerçek dünya görüntüsü ve sınıf içeren büyük görüntü veri kümelerinden biridir.

Bir model ImageNet üzerinde eğitim aldığında:

- kenar,
- doku,
- şekil,
- nesne parçası

gibi görsel temsilleri öğrenmiş olabilir.

Bu özellikler başka görüntü problemlerine aktarılabilir.


# 60. MobileNetV2 Neden Uygun?

MobileNet ailesi özellikle daha düşük hesaplama maliyetini hedefleyen görüntü modellerindendir.

Bu nedenle:

- eğitim,
- mobil uygulama,
- gömülü sistem,
- hızlı prototip

çalışmalarında yararlı olabilir.

Bu derste küçük giriş boyutu kullanarak yapıyı inceleyeceğiz.


# 61. MobileNetV2 Girdi Şekli

MobileNetV2:

```text
3 kanallı görüntü
```

bekler.

`include_top=False` kullanıldığında farklı uzamsal boyutlarla çalışılabilir.

Bu derste:

```text
96 × 96 × 3
```

kullanacağız.


# 62. ImageNet Ağırlıklarını Güvenli Şekilde Yüklemek

İlk çalıştırmada ağırlıkların indirilebilmesi için internet bağlantısı gerekebilir.

Notebook internet erişimi yoksa dersin ilk yerel transfer learning bölümü yine çalışmaya devam eder.

Aşağıdaki kod indirme başarısız olursa mimariyi `weights=None` ile oluşturur ve ImageNet'e dayalı eğitim bölümünü atlar.


In [ ]:
IMAGENET_HAZIR = False

try:
    mobilenet_base = (
        keras.applications.MobileNetV2(
            input_shape=(
                96,
                96,
                3
            ),
            include_top=False,
            weights="imagenet"
        )
    )

    IMAGENET_HAZIR = True

    print(
        "ImageNet ağırlıkları yüklendi."
    )

except Exception as hata:
    print(
        "ImageNet ağırlıkları yüklenemedi."
    )

    print(
        "Mimari weights=None ile oluşturuluyor."
    )

    mobilenet_base = (
        keras.applications.MobileNetV2(
            input_shape=(
                96,
                96,
                3
            ),
            include_top=False,
            weights=None
        )
    )

print(
    "IMAGENET_HAZIR:",
    IMAGENET_HAZIR
)


# 63. MobileNetV2 Model Özeti

Tam model özeti oldukça uzundur.

Katman sayısını ve parametre sayısını inceleyelim.


In [ ]:
print(
    "Katman sayısı:",
    len(
        mobilenet_base.layers
    )
)

print(
    "Parametre sayısı:",
    f"{mobilenet_base.count_params():,}"
)


# 64. `include_top=False` Ne Demektir?

ImageNet modelinin orijinal 1000 sınıflı classification head'ini istemediğimizi belirtir.

Yani:

```text
MobileNetV2 Feature Extractor
↓
Bizim Yeni Head
```

kurabiliriz.


# 65. MobileNetV2 Ön İşleme

Her Keras Application modeli aynı piksel ön işleme kuralını kullanmak zorunda değildir.

MobileNetV2 için:

```python
keras.applications.mobilenet_v2.preprocess_input
```

kullanılır.

Bu işlem giriş piksel değerlerini modelin eğitim sırasında beklediği ölçeğe dönüştürür.


In [ ]:
ornek_pixel = np.array(
    [
        0.0,
        127.5,
        255.0
    ],
    dtype=np.float32
)

print(
    keras.applications.mobilenet_v2.preprocess_input(
        ornek_pixel.copy()
    )
)


MobileNetV2 preprocessing yaklaşık olarak piksel değerlerini `-1` ile `1` aralığına ölçekler.


# 66. Sentetik Şekil Veri Kümesi

MobileNetV2 transfer learning yapısını göstermek için dış veri indirmeden:

- daire,
- kare,
- üçgen

görüntüleri oluşturacağız.

ImageNet ağırlıkları varsa pretrained feature extractor kullanacağız.

Bu veri yalnızca öğretim amaçlıdır.


In [ ]:
import cv2

rng = np.random.default_rng(
    42
)

SINIF_ADLARI = [
    "Daire",
    "Kare",
    "Ucgen"
]

def sekil_goruntusu_uret(
    sinif,
    boyut=96
):
    image = np.full(
        (
            boyut,
            boyut,
            3
        ),
        rng.integers(
            5,
            40
        ),
        dtype=np.uint8
    )

    merkez_x = int(
        rng.integers(
            38,
            59
        )
    )

    merkez_y = int(
        rng.integers(
            38,
            59
        )
    )

    olcek = int(
        rng.integers(
            18,
            30
        )
    )

    renk = tuple(
        int(x)
        for x in rng.integers(
            100,
            256,
            size=3
        )
    )

    if sinif == 0:
        cv2.circle(
            image,
            (
                merkez_x,
                merkez_y
            ),
            olcek,
            renk,
            -1
        )

    elif sinif == 1:
        cv2.rectangle(
            image,
            (
                merkez_x - olcek,
                merkez_y - olcek
            ),
            (
                merkez_x + olcek,
                merkez_y + olcek
            ),
            renk,
            -1
        )

    else:
        pts = np.array([
            [
                merkez_x,
                merkez_y - olcek
            ],
            [
                merkez_x - olcek,
                merkez_y + olcek
            ],
            [
                merkez_x + olcek,
                merkez_y + olcek
            ]
        ])

        cv2.fillPoly(
            image,
            [
                pts
            ],
            renk
        )

    noise = rng.normal(
        0,
        4,
        image.shape
    )

    image = np.clip(
        image.astype(
            np.float32
        )
        +
        noise,
        0,
        255
    ).astype(
        np.uint8
    )

    return image


# 67. Şekil Veri Kümesini Oluşturmak

In [ ]:
sekil_images = []
sekil_labels = []

for sinif in range(3):
    for _ in range(100):
        sekil_images.append(
            sekil_goruntusu_uret(
                sinif
            )
        )

        sekil_labels.append(
            sinif
        )

sekil_images = np.array(
    sekil_images
)

sekil_labels = np.array(
    sekil_labels
)

print(
    sekil_images.shape,
    sekil_labels.shape
)


# 68. Örnek Şekilleri Görmek

In [ ]:
for sinif in range(3):
    index = np.where(
        sekil_labels
        ==
        sinif
    )[0][0]

    plt.figure()

    plt.imshow(
        cv2.cvtColor(
            sekil_images[
                index
            ],
            cv2.COLOR_BGR2RGB
        )
    )

    plt.title(
        SINIF_ADLARI[
            sinif
        ]
    )

    plt.axis("off")
    plt.show()


# 69. BGR'den RGB'ye Dönüştürmek

OpenCV görüntülerimizi BGR oluşturduğumuz için model girişinden önce RGB sırasına çevirelim.


In [ ]:
sekil_rgb = np.array([
    cv2.cvtColor(
        image,
        cv2.COLOR_BGR2RGB
    )
    for image in sekil_images
])

sekil_rgb = sekil_rgb.astype(
    np.float32
)

print(
    sekil_rgb.shape
)


# 70. Train-Test Ayrımı

In [ ]:
X_shape_train, X_shape_test, y_shape_train, y_shape_test = train_test_split(
    sekil_rgb,
    sekil_labels,
    test_size=0.25,
    random_state=42,
    stratify=sekil_labels
)

print(
    X_shape_train.shape,
    X_shape_test.shape
)


# 71. MobileNetV2 Base Modeli Dondurmak

In [ ]:
mobilenet_base.trainable = False

print(
    mobilenet_base.trainable
)


# 72. Data Augmentation

Şekillerde küçük dönüş ve kaydırmalar sınıfı değiştirmeyeceği için augmentation kullanabiliriz.


In [ ]:
shape_augmentation = keras.Sequential([
    layers.RandomRotation(
        0.05
    ),
    layers.RandomTranslation(
        0.05,
        0.05
    )
])


# 73. MobileNetV2 Transfer Modeli

Functional API ile:

```text
Input
↓
Augmentation
↓
MobileNetV2 preprocess_input
↓
Frozen MobileNetV2
↓
GlobalAveragePooling2D
↓
Dropout
↓
Dense(3)
```

kuracağız.


In [ ]:
inputs = keras.Input(
    shape=(
        96,
        96,
        3
    )
)

x = shape_augmentation(
    inputs
)

x = keras.applications.mobilenet_v2.preprocess_input(
    x
)

x = mobilenet_base(
    x,
    training=False
)

x = layers.GlobalAveragePooling2D()(
    x
)

x = layers.Dropout(
    0.25
)(
    x
)

outputs = layers.Dense(
    3,
    activation="softmax"
)(
    x
)

mobilenet_transfer = keras.Model(
    inputs,
    outputs,
    name="mobilenet_transfer"
)

mobilenet_transfer.summary()


`mobilenet_base(x, training=False)` kullanımı özellikle BatchNormalization katmanlarının transfer learning davranışını kontrollü tutmak için önemlidir.


# 74. Trainable Parametre Sayısı

In [ ]:
trainable_params = sum(
    int(
        np.prod(
            w.shape
        )
    )
    for w
    in mobilenet_transfer.trainable_weights
)

non_trainable_params = sum(
    int(
        np.prod(
            w.shape
        )
    )
    for w
    in mobilenet_transfer.non_trainable_weights
)

print(
    "Trainable:",
    f"{trainable_params:,}"
)

print(
    "Non-trainable:",
    f"{non_trainable_params:,}"
)


Frozen feature extractor sayesinde milyonlarca parametrenin tamamını yeniden eğitmek zorunda değiliz.


# 75. MobileNet Transfer Modelini Compile Etmek

In [ ]:
mobilenet_transfer.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy"
    ]
)


# 76. ImageNet Varsa Transfer Eğitimini Çalıştırmak

Ağırlıklar indirilemediyse `weights=None` modeli transfer learning değildir.

Bu nedenle gerçek transfer eğitimini yalnızca `IMAGENET_HAZIR=True` olduğunda çalıştırıyoruz.


In [ ]:
mobilenet_history = None

if IMAGENET_HAZIR:
    mobilenet_history = (
        mobilenet_transfer.fit(
            X_shape_train,
            y_shape_train,
            validation_split=0.20,
            epochs=12,
            batch_size=16,
            callbacks=[
                keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=3,
                    restore_best_weights=True
                )
            ],
            verbose=0
        )
    )

    print(
        "ImageNet transfer eğitimi tamamlandı."
    )

else:
    print(
        "ImageNet ağırlıkları olmadığı için gerçek pretrained eğitim atlandı."
    )


# 77. MobileNet Learning Curve

In [ ]:
if mobilenet_history is not None:
    plt.plot(
        mobilenet_history.history[
            "accuracy"
        ],
        label="Train"
    )

    plt.plot(
        mobilenet_history.history[
            "val_accuracy"
        ],
        label="Validation"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("MobileNetV2 Transfer Learning")
    plt.legend()
    plt.show()

else:
    print(
        "Learning curve için ImageNet ağırlıkları gereklidir."
    )


# 78. MobileNet Final Test

In [ ]:
if IMAGENET_HAZIR:
    mobilenet_loss, mobilenet_accuracy = (
        mobilenet_transfer.evaluate(
            X_shape_test,
            y_shape_test,
            verbose=0
        )
    )

    print(
        "Test Accuracy:",
        mobilenet_accuracy
    )

else:
    mobilenet_accuracy = None

    print(
        "Pretrained test değerlendirmesi atlandı."
    )


# 79. MobileNet Tahminleri

In [ ]:
if IMAGENET_HAZIR:
    shape_prob = (
        mobilenet_transfer.predict(
            X_shape_test,
            verbose=0
        )
    )

    shape_pred = np.argmax(
        shape_prob,
        axis=1
    )

    print(
        classification_report(
            y_shape_test,
            shape_pred,
            target_names=SINIF_ADLARI
        )
    )

else:
    print(
        "ImageNet ağırlıkları bulunmadığı için tahmin bölümü atlandı."
    )


# 80. MobileNet Confusion Matrix

In [ ]:
if IMAGENET_HAZIR:
    shape_cm = confusion_matrix(
        y_shape_test,
        shape_pred
    )

    ConfusionMatrixDisplay(
        confusion_matrix=shape_cm,
        display_labels=SINIF_ADLARI
    ).plot()

    plt.title(
        "MobileNetV2 Transfer Learning"
    )

    plt.show()

else:
    print(
        "Confusion matrix için pretrained model eğitimi gereklidir."
    )


# 81. Gerçek Önceden Eğitilmiş Modelde Fine-Tuning

Head eğitimi tamamlandıktan sonra base modelin son katmanlarının bir bölümünü açabiliriz.

Ancak:

- çok düşük learning rate,
- validation takibi,
- BatchNormalization katmanlarına dikkat

gereklidir.


# 82. Son 20 Katmanı Fine-Tune Etmeye Hazırlamak

In [ ]:
if IMAGENET_HAZIR:
    mobilenet_base.trainable = True

    for layer in (
        mobilenet_base.layers[:-20]
    ):
        layer.trainable = False

    for layer in (
        mobilenet_base.layers
    ):
        if isinstance(
            layer,
            layers.BatchNormalization
        ):
            layer.trainable = False

    print(
        "Fine-tuning için katmanlar ayarlandı."
    )

else:
    print(
        "ImageNet ağırlıkları olmadan fine-tuning uygulanmayacak."
    )


# 83. Trainable Katman Sayısı

In [ ]:
if IMAGENET_HAZIR:
    trainable_layer_count = sum(
        1
        for layer
        in mobilenet_base.layers
        if layer.trainable
    )

    print(
        "Trainable base layer:",
        trainable_layer_count
    )


# 84. Fine-Tuning İçin Yeniden Compile

`trainable` durumu değiştiği için model yeniden compile edilir.


In [ ]:
if IMAGENET_HAZIR:
    mobilenet_transfer.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=1e-5
        ),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy"
        ]
    )

    print(
        "Model düşük learning rate ile yeniden compile edildi."
    )


# 85. MobileNet Fine-Tuning

In [ ]:
mobilenet_fine_history = None

if IMAGENET_HAZIR:
    mobilenet_fine_history = (
        mobilenet_transfer.fit(
            X_shape_train,
            y_shape_train,
            validation_split=0.20,
            epochs=5,
            batch_size=16,
            callbacks=[
                keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=2,
                    restore_best_weights=True
                )
            ],
            verbose=0
        )
    )

    fine_test_loss, fine_test_accuracy = (
        mobilenet_transfer.evaluate(
            X_shape_test,
            y_shape_test,
            verbose=0
        )
    )

    print(
        "Fine-Tuned Test Accuracy:",
        fine_test_accuracy
    )

else:
    fine_test_accuracy = None

    print(
        "Fine-tuning atlandı."
    )


# 86. Frozen ve Fine-Tuned Sonuç

Fine-tuning her zaman iyileşme sağlamaz.

Küçük veri kümelerinde overfitting hızlı başlayabilir.


In [ ]:
if IMAGENET_HAZIR:
    mobilenet_karsilastirma = pd.DataFrame({
        "Asama": [
            "Frozen MobileNetV2",
            "Fine-Tuned MobileNetV2"
        ],
        "TestAccuracy": [
            mobilenet_accuracy,
            fine_test_accuracy
        ]
    })

    print(
        mobilenet_karsilastirma
    )


# 87. MobileNet Modelini Kaydetmek

In [ ]:
if IMAGENET_HAZIR:
    mobilenet_transfer.save(
        "32-mobilenet-transfer-modeli.keras"
    )

    print(
        "MobileNet transfer modeli kaydedildi."
    )

else:
    print(
        "Pretrained model olmadığı için kayıt bölümü atlandı."
    )


# 88. MobileNet Tahmin Fonksiyonu

In [ ]:
def mobilenet_sekil_tahmin(
    model,
    rgb_goruntu
):
    image = np.array(
        rgb_goruntu,
        dtype=np.float32
    )

    if image.shape != (
        96,
        96,
        3
    ):
        image = tf.image.resize(
            image,
            (
                96,
                96
            )
        ).numpy()

    image = image.reshape(
        1,
        96,
        96,
        3
    )

    probability = model.predict(
        image,
        verbose=0
    )[0]

    prediction = int(
        np.argmax(
            probability
        )
    )

    return {
        "sinif": prediction,
        "etiket":
            SINIF_ADLARI[
                prediction
            ],
        "skorlar":
            probability
    }


Modelin preprocessing adımı Functional modelin içinde bulunduğu için tahmin fonksiyonunda ayrıca `preprocess_input()` çağırmadık.


# 89. Tahmin Fonksiyonunu Test Etmek

In [ ]:
if IMAGENET_HAZIR:
    sonuc = mobilenet_sekil_tahmin(
        mobilenet_transfer,
        X_shape_test[0]
    )

    print(
        "Gerçek:",
        SINIF_ADLARI[
            y_shape_test[0]
        ]
    )

    print(
        "Tahmin:",
        sonuc[
            "etiket"
        ]
    )

else:
    print(
        "Pretrained model eğitimi yapılmadığı için test atlandı."
    )


# 90. Feature Extraction mı Fine-Tuning mi?

### Önce Feature Extraction

Avantajlar:

- hızlı,
- daha az trainable parametre,
- küçük veride daha güvenli,
- önceden öğrenilmiş özellikler korunur.

### Sonra Gerekirse Fine-Tuning

Avantaj:

- base özellikleri target probleme biraz uyarlayabilir.

Risk:

- overfitting,
- önceden öğrenilmiş ağırlıkları bozma,
- daha uzun eğitim.


# 91. Hangi Katmanlar Fine-Tune Edilmeli?

Kesin bir sayı yoktur.

Genel yaklaşım:

- önce tüm base modeli dondur,
- yeni head'i eğit,
- validation sonucunu incele,
- son katmanlardan küçük bir bölümü aç,
- düşük learning rate kullan.

Target veri source veriye çok benziyorsa daha az katman açmak yeterli olabilir.


# 92. Source ve Target Benzerliği

Transfer learning'in başarısında source ve target problem ilişkisi önemlidir.

Örnek:

```text
Source → genel nesne fotoğrafları
Target → hayvan fotoğrafları
```

görsel özellikler açısından oldukça ilişkili olabilir.

Ancak tamamen farklı veri türleri arasında aktarım daha sınırlı olabilir.


# 93. Negative Transfer

Transfer edilen bilgi yeni probleme yardımcı olmak yerine performansı kötüleştirirse buna **negative transfer** denebilir.

Bu nedenle:

```text
pretrained model kullandım
=
kesin daha iyi sonuç
```

değildir.

Transfer modeli mutlaka baseline veya scratch modelle karşılaştırılmalıdır.


# 94. Pretrained Model Seçimi

Model seçerken yalnızca accuracy düşünülmez.

Ayrıca:

- parametre sayısı,
- giriş boyutu,
- tahmin hızı,
- model dosyası boyutu,
- cihaz kapasitesi,
- lisans,
- hedef platform

gibi faktörler değerlendirilir.


# 95. MobileNet ve Mobil Uygulamalar

MobileNet ailesi daha verimli görüntü modelleri hedeflediği için mobil ve gömülü uygulamalarda sık değerlendirilen mimariler arasındadır.

Ancak gerçek cihaz performansı:

- model sürümü,
- giriş boyutu,
- quantization,
- donanım,
- runtime

gibi faktörlere bağlıdır.


# 96. Transfer Learning ve Data Augmentation

Küçük target veri kümelerinde augmentation özellikle yararlı olabilir.

Ancak dönüşüm:

```text
etiketi değiştirmemeli
```

kuralına dikkat etmeliyiz.

Örneğin bazı rakam veya yön problemlerinde flip kullanmak hatalı veri üretebilir.


# 97. BatchNormalization ve Fine-Tuning

Önceden eğitilmiş modellerde BatchNormalization katmanları özel dikkat gerektirir.

Fine-tuning sırasında bu katmanların istatistiklerinin küçük target veriyle hızlı biçimde bozulması istenmeyebilir.

Bu nedenle örneğimizde BatchNormalization katmanlarını dondurduk ve base modeli çağırırken:

```python
training=False
```

kullandık.


# 98. Transfer Learning'de Veri Sızıntısı

Source modelin target test verisini eğitim sırasında görmemesi gerekir.

Ayrıca target model seçimi yaparken:

```text
Final Test
```

sürekli kullanılmamalıdır.

Train / validation / test ayrımı transfer learning projelerinde de geçerlidir.


# 99. Ön İşleme Model Kadar Önemlidir

Önceden eğitilmiş model belirli bir preprocessing ile eğitilmiş olabilir.

Örneğin MobileNetV2:

```python
mobilenet_v2.preprocess_input
```

bekler.

Yanlış piksel ölçeği veya kanal sırası modeli ciddi biçimde etkileyebilir.


# 100. BGR ve RGB Dikkati

OpenCV:

```text
BGR
```

Keras Applications ve yaygın görüntü veri akışları:

```text
RGB
```

kullanabilir.

Gerçek uygulamada:

```python
cv2.cvtColor(
    image,
    cv2.COLOR_BGR2RGB
)
```

dönüşümü gerekebilir.


# 101. Transfer Learning'de Model Boyutu

Büyük pretrained modeller milyonlarca parametre içerebilir.

Ancak base model frozen olduğunda eğitim sırasında bütün parametrelerin gradienti hesaplanmaz.

Bu transfer eğitimini sıfırdan tam model eğitimine göre daha ekonomik hale getirebilir.


# 102. Feature Cache Yaklaşımı

Dondurulmuş base model için:

```text
Tüm görüntüleri bir kez base modelden geçir
↓
Feature'ları kaydet
↓
Yeni küçük classifier eğit
```

yaklaşımı çok hızlı olabilir.

Ancak dinamik data augmentation uygulamak isteniyorsa base modeli eğitim akışında tutmak daha uygundur.


# 103. Fine-Tuning'de Learning Rate

Head eğitimi:

```text
1e-3
```

gibi normal bir learning rate ile başlayabilir.

Fine-tuning:

```text
1e-5
```

gibi çok daha küçük learning rate ile yapılabilir.

Kesin değer problem ve optimizer'a göre değişir.


# 104. Transfer Learning Uygulama Akışı

```text
Problem
↓
Target Veri Kümesi
↓
Pretrained Base Model
↓
include_top=False
↓
Doğru Preprocessing
↓
Base Modeli Freeze Et
↓
Yeni Head Ekle
↓
Head'i Eğit
↓
Validation
↓
Gerekirse Son Katmanları Aç
↓
Düşük Learning Rate
↓
Fine-Tuning
↓
Final Test
↓
Model Kaydı
```


# 105. Gerçek Projede Kullanılabilecek Pretrained Modeller

Keras Applications içinde çeşitli mimariler bulunur.

Örnekler:

```text
MobileNetV2
MobileNetV3
ResNet50
EfficientNetB0
DenseNet121
Xception
InceptionV3
```

Her modelin:

- giriş boyutu,
- preprocessing yöntemi,
- parametre sayısı,
- performansı

farklıdır.

Model dokümantasyonu mutlaka kontrol edilmelidir.


# 106. Transfer Learning ile Nesne Tespiti Aynı Şey mi?

Hayır.

Transfer learning bir **öğrenme stratejisidir**.

Nesne tespiti ise bir görevdir.

Transfer learning:

- görüntü sınıflandırma,
- nesne tespiti,
- segmentasyon,
- NLP,
- ses işleme

gibi birçok görevde kullanılabilir.


# 107. CNN'den LLM'e Transfer Learning

Transfer learning yalnızca görüntü alanına özgü değildir.

Modern doğal dil işlemede de büyük dil modelleri:

```text
ön eğitim
↓
genel dil özellikleri
↓
prompting / fine-tuning / adaptation
↓
özel görev
```

yaklaşımından yararlanır.

Bu fikir bizi ilerideki üretken yapay zeka derslerine bağlayacaktır.


# 108. Model Dosyası Güvenliği

Model dosyalarını yalnızca güvenilen kaynaklardan kullanmalıyız.

Özellikle model:

- özel katman,
- Python serialization,
- harici bağımlılık

içeriyorsa güvenlik ve uyumluluk kontrol edilmelidir.


# 109. Etik ve Veri Lisansı

Pretrained model kullanırken yalnızca teknik özellikler değil:

- model lisansı,
- eğitim verisinin kullanım koşulları,
- target veri izinleri,
- kişisel veri,
- kullanım alanı

da değerlendirilmelidir.


# 110. Model Bias

Pretrained model source veri kümesindeki örüntüleri ve olası önyargıları da taşıyabilir.

Transfer learning:

```text
yalnızca yararlı özellikleri
```

değil, bazı veri kaynaklı sınırlılıkları da aktarabilir.

Gerçek uygulamada target veri üzerinde ayrı performans ve hata analizi yapılmalıdır.


# 111. Transfer Learning Başarı Kontrol Listesi

Bir projede şu soruları sorabiliriz:

- Source ve target problem ilişkili mi?
- Doğru pretrained model seçildi mi?
- Doğru preprocessing kullanılıyor mu?
- Base model önce donduruldu mu?
- Yeni head yeterince eğitildi mi?
- Fine-tuning gerekiyorsa learning rate düşük mü?
- `trainable` değişince yeniden compile edildi mi?
- BatchNormalization davranışı kontrol edildi mi?
- Scratch baseline ile karşılaştırıldı mı?
- Final test ayrı tutuldu mu?


# 112. Ders Özeti

Bu derste:

- Transfer Learning,
- source task,
- target task,
- pretrained model,
- feature extractor,
- feature extraction,
- classification head,
- `trainable=False`,
- frozen layer,
- trainable weights,
- scratch model,
- pretrained-scratch karşılaştırması,
- feature cache,
- fine-tuning,
- düşük learning rate,
- yeniden compile,
- MobileNetV2,
- ImageNet,
- `include_top=False`,
- modele özel preprocessing,
- GlobalAveragePooling2D,
- BatchNormalization davranışı,
- data augmentation,
- negative transfer,
- model seçimi,
- `.keras` model kaydı

konularını öğrendik.


# 113. Mini Uygulamalar

1. Digits veri kümesini yükleyin.
2. Bir CNN feature extractor oluşturun.
3. 10 sınıflı source model eğitin.
4. Feature extractor çıktı şeklini inceleyin.
5. Feature extractor'ın feature map'ini gösterin.
6. Target etiketleri tek/çift olarak oluşturun.
7. Target eğitim verisini 200 örnekle sınırlandırın.
8. Feature extractor'ı dondurun.
9. Trainable ve non-trainable weight sayılarını bulun.
10. Yeni sigmoid classification head oluşturun.
11. Frozen transfer modelini eğitin.
12. Confusion matrix oluşturun.
13. Aynı target problem için sıfırdan CNN eğitin.
14. Scratch ve transfer accuracy'lerini karşılaştırın.
15. Feature'ları önceden hesaplayın.
16. CNN feature'larıyla Logistic Regression eğitin.
17. Base modelin son katmanlarını açın.
18. Learning rate'i düşürerek fine-tuning yapın.
19. Fine-tuning öncesi ve sonrası sonucu karşılaştırın.
20. Transfer modelini `.keras` olarak kaydedin.
21. MobileNetV2'yi `include_top=False` ile oluşturun.
22. MobileNetV2 preprocessing sonucunu inceleyin.
23. Pretrained base modeli dondurun.
24. GlobalAveragePooling2D + Dense head ekleyin.
25. ImageNet erişimi varsa küçük bir transfer learning uygulaması çalıştırın.


# 114. Yapay Zeka Proje Görevi

Bir **Transfer Learning Görüntü Sınıflandırma Projesi** geliştirin.

Konu seçenekleri:

- üç farklı geometrik şekil,
- üç ürün kategorisi,
- bitki yaprağı sınıfları,
- araç türleri,
- nesne kategorileri.

Projede en az:

- en az 3 sınıf,
- train / validation / test ayrımı,
- data augmentation,
- bir scratch CNN baseline,
- bir pretrained model,
- `include_top=False`,
- frozen base,
- yeni classification head,
- doğru preprocessing,
- feature extraction eğitimi,
- accuracy,
- classification report,
- confusion matrix,
- yanlış tahmin analizi,
- fine-tuning,
- düşük learning rate,
- scratch-transfer karşılaştırması,
- `.keras` model kaydı

bulunsun.

Gerçek görüntü kullanılıyorsa veri kullanım izinleri ve kişisel veri konuları ayrıca değerlendirilmelidir.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin aşağıdaki transfer learning zincirini kurabilmesi hedeflenmektedir:

**Önceden Eğitilmiş Model**

↓

**Feature Extractor**

↓

**Freeze**

↓

**Yeni Target Veri**

↓

**Yeni Classification Head**

↓

**Feature Extraction Eğitimi**

↓

**Validation**

↓

**Gerekirse Unfreeze**

↓

**Düşük Learning Rate**

↓

**Fine-Tuning**

↓

**Final Test**

↓

**Model Kaydı**

Artık öğrenciler yalnızca kendi CNN'lerini sıfırdan eğitmekle kalmıyor; daha önce öğrenilmiş görsel özellikleri yeni problemlere aktararak çok daha güçlü yapay zeka geliştirme stratejilerini kullanabiliyor.

Bir sonraki derste görüntü modellerinden üretken yapay zekaya geçecek ve **Büyük Dil Modelleri, token, context, prompt ve üretken yapay zeka uygulamalarının temel çalışma mantığını** inceleyeceğiz.
